In [ ]:
import QuantLib as ql
import numpy as np
import pandas as pd
import random as rd
import os
import multiprocessing as mp
import time
import tqdm

### Generate Correlation Matrix

In [ ]:
def createCorrMatrix(d,betaparam1, betaparam2):

    P = np.zeros(shape=(d,d),dtype=float)

    S = ql.Matrix(d,d,0)
    for j in range(d):
        S[j][j] = 1.0

    for k in range (d-1):
        for i in range(k+1,d):
            P[k][i] = (np.random.beta(betaparam1,betaparam2) - 0.5) * 2
            p = P[k][i]
            for l in range(k-1,1,-1):
                p = p * np.sqrt((1-P[l][i]**2)*(1-P[l][k]**2)) + P[l][i]*P[l][k]
            S[k][i] = p
            S[i][k] = p
    
    return(S)

### Generate Basket Data

In [ ]:
def generateData(ID, 
                 runs, 
                 filename,
                 basketSize = 6, 
                 strike = 100.0, 
                 interestRate=0.0, 
                 sampleSizes = [10000, ]):

    df1 = pd.DataFrame()
    for run in range(runs):

        todaysDate = ql.Date(13, 4, 2018)
        ql.Settings.instance().evaluationDate = todaysDate

        maturity = int(rd.uniform(1,43)**2.0)

        stocks = [100 * rd.lognormvariate(0.5,0.25) for i in range(basketSize)]
        vols = [rd.uniform(0.0,1.0) for i in range(basketSize)]
        divs = [0 for i in range(basketSize)]

        settlementDate = todaysDate 
        riskFreeRate = ql.FlatForward(settlementDate, interestRate, ql.Actual365Fixed())

        # option parameters
        exercise = ql.EuropeanExercise(todaysDate + maturity)
        payoff = ql.PlainVanillaPayoff(ql.Option.Call, strike)

        # market data
        underlying = [ql.SimpleQuote(stock) for stock in stocks]
        volatility = [ql.BlackConstantVol(todaysDate, ql.TARGET(), vol, ql.Actual365Fixed()) for vol in vols]
        dividendYield = [ql.FlatForward(settlementDate, div, ql.Actual365Fixed()) for div in divs]

        processes = [ql.BlackScholesMertonProcess(ql.QuoteHandle(underlying[i]),\
                              ql.YieldTermStructureHandle(dividendYield[i]),\
                              ql.YieldTermStructureHandle(riskFreeRate),\
                              ql.BlackVolTermStructureHandle(volatility[i])) for i in range(basketSize)]

        betaparam1 = 5 #set between 1-50. Low number makes for high pairwise correlations
        betaparam2 = 2 #set between 1-50. Low number makes for high pairwise correlations
        corrMatrix = createCorrMatrix(basketSize,betaparam1,betaparam2)
        process = ql.StochasticProcessArray(processes, corrMatrix)
        basketoption = ql.BasketOption(ql.MinBasketPayoff(payoff), exercise)


        for sampleSize in sampleSizes:

            basketEngine = ql.MCEuropeanBasketEngine(process,'PseudoRandom', timeSteps = 1,\
                                           requiredSamples = sampleSize, maxSamples = sampleSize)
            basketoption.setPricingEngine(basketEngine)

            startTime = time.process_time()
            #print(startTime)
            try:
                optionValue = basketoption.NPV()
                errorEstimate = basketoption.errorEstimate()

                endTime = time.process_time()
                elapsedTime = endTime - startTime

                mydata= {'maturity':[float(maturity)]}

                for i in range(basketSize):
                    stockName = 'stockPrice' + str(i)
                    volName = 'vols' + str(i)
                    mydata[stockName] = stocks[i]
                    mydata[volName] = vols[i]

                for j in range(basketSize-1):
                    for k in range(j+1,basketSize):
                        rhoName = 'rho_' + str(j) + '_' + str(k)
                        mydata[rhoName] = corrMatrix[j][k]

                mydata['optionValue'] = [float(optionValue)]
                mydata['errorEstimate'] = [float(errorEstimate)]
                mydata['samples'] = [int(sampleSize)]
                mydata['processTime'] = [float(elapsedTime)]

                if run == 0:
                    df1 = pd.DataFrame(data=mydata)
                else:
                    df2 = pd.DataFrame(data=mydata)
                    df1 = df1.append(df2, ignore_index=True)

                csvfile = filename + str(ID) + '.csv'

            except Exception as myException:
                endTime = time.process_time()
                elapsedTime = endTime - startTime
                print("Error: {0}".format(myException))

    df1.to_csv(csvfile)
    print(ID)
    return(ID)

### Configure Run & Execute

In [ ]:
filename = 'basket_option100000x10'
no_threads = os.cpu_count() - 2
pkgsize = 100000
dataSize = 10

In [ ]:
rd.seed()
pool = mp.Pool(processes=no_threads)
pbar = tqdm.tqdm(total=no_threads)
results = [pool.apply_async(generateData, args=(i,pkgsize,filename), 
                            callback=lambda _: pbar.update(1)) for i in range(dataSize)]
output = [p.get() for p in results]